In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta import DeltaTable
dbutils.widgets.text("incremental_flag","0")
incremental_flag = dbutils.widgets.get("incremental_flag")
df = spark.sql("select * from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver`").display()

In [0]:
df_src = spark.sql("select distinct Dealer_ID,DealerName from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver`")


if spark.catalog.tableExists("car_sales.gold.dim_dealer"):
    df_sink = spark.sql("select Dealer_ID,DealerName,dim_dealer_key from car_sales.gold.dim_dealer")
else:
    df_sink = spark.sql("select Dealer_ID,DealerName,0 as dim_dealer_key from parquet.`abfss://carsales@adls7786.dfs.core.windows.net/silver` where 1=0")
df_sink.display()

df_filter = df_src.join(df_sink,df_src['Dealer_ID'] == df_sink['Dealer_ID'],'left').select(df_src['Dealer_ID'],df_src['DealerName'],df_sink['dim_dealer_key'])

df_filter_old = df_filter.where(df_filter['dim_dealer_key'].isNotNull())
df_filter_new = df_filter.where(df_filter['dim_dealer_key'].isNull()).select("DealerName","Dealer_ID")


if (incremental_flag == "0"):
    max_value = 1
else:
    if spark.catalog.tableExists("car_sales.gold.dim_dealer"):
        df = spark.sql("select max(dim_dealer_key) from car_sales.gold.dim_dealer")
        max_value = df.collect()[0][0]

df_filter_new = df_filter_new.withColumn("dim_dealer_key",F.lit(max_value) + F.monotonically_increasing_id())

df_filter_new.display()

df_final = df_filter_new.union(df_filter_old)

if spark.catalog.tableExists("car_sales.gold.dim_dealer"):
    delta_table = DeltaTable.forPath(spark,'abfss://carsales@adls7786.dfs.core.windows.net/gold/dim_dealer')
    delta_table.alias("t").merge(df_final.alias("s"),"t.dim_dealer_key = s.dim_dealer_key").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format("delta").mode("overwrite").option("path","abfss://carsales@adls7786.dfs.core.windows.net/gold/dim_dealer").saveAsTable("car_sales.gold.dim_dealer")


In [0]:
%sql

select * from car_sales.gold.dim_dealer;